In [ ]:
cat > /home/claude/chaguoai_model/notebooks/chaguoai_model_training.ipynb << 'NOTEBOOK_EOF'
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# ChaguoAI — Contraceptive Discontinuation Prediction Model\n",
    "\n",
    "**Purpose:** Predict which women are at highest risk of discontinuing their chosen contraceptive method so ChaguoAI can prioritize follow-up messages and recommend methods with the highest predicted adherence.\n",
    "\n",
    "**Datasets used:**\n",
    "- `Client_Service_Statistics.csv` — 216,539 FP service delivery records (Siaya & Busia, Kenya, 2013–2015). **Primary training dataset.**\n",
    "- `Final_women_Data_ANON.csv` — 1,997 Western Kenya women survey. **EDA and clinical validation only.**\n",
    "\n",
    "**Why CSS is the primary dataset:** It records direct observed behaviour — a woman came in on a prior method and left on the same or different method. That switch is a real, unrecalled discontinuation signal. The women survey has only 514 ever-users with a discontinuation signal — too small for reliable ML training.\n",
    "\n",
    "**Data split:** 70% train / 15% validation / 15% test (all stratified on target)\n",
    "\n",
    "---\n",
    "## Notebook Structure\n",
    "1. Environment Setup\n",
    "2. Data Loading and Validation\n",
    "3. Exploratory Data Analysis (12 charts)\n",
    "4. Feature Engineering\n",
    "5. Target Variable Construction\n",
    "6. Encoding and Final Feature Matrix\n",
    "7. 70/15/15 Data Split\n",
    "8. Class Imbalance Handling\n",
    "9. Model Training and Cross-Validation\n",
    "10. Threshold Optimisation on Validation Set\n",
    "11. Final Evaluation on Test Set\n",
    "12. Feature Importance and SHAP\n",
    "13. Model Persistence\n",
    "14. Inference Function\n",
    "15. Scalability and Retraining Architecture\n",
    "16. Design Decisions and Documentation"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Environment Setup"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import subprocess, sys\n",
    "\n",
    "def ensure_package(package):\n",
    "    try:\n",
    "        __import__(package.split('==')[0].replace('-','_'))\n",
    "    except ImportError:\n",
    "        subprocess.check_call([sys.executable,'-m','pip','install',package,'-q'])\n",
    "\n",
    "for pkg in ['scikit-learn','xgboost','lightgbm','imbalanced-learn','shap',\n",
    "            'matplotlib','seaborn','pandas','numpy']:\n",
    "    ensure_package(pkg)\n",
    "\n",
    "print('All packages ready.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, json, pickle, warnings\n",
    "from pathlib import Path\n",
    "from datetime import datetime\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "import seaborn as sns\n",
    "\n",
    "from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score\n",
    "from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "from sklearn.metrics import (\n",
    "    roc_auc_score, average_precision_score, classification_report,\n",
    "    confusion_matrix, precision_recall_curve, RocCurveDisplay, ConfusionMatrixDisplay\n",
    ")\n",
    "from imblearn.over_sampling import SMOTE\n",
    "import xgboost as xgb\n",
    "import lightgbm as lgb\n",
    "\n",
    "try:\n",
    "    import shap\n",
    "    SHAP_AVAILABLE = True\n",
    "except ImportError:\n",
    "    SHAP_AVAILABLE = False\n",
    "\n",
    "warnings.filterwarnings('ignore')\n",
    "pd.set_option('display.max_columns', 50)\n",
    "np.random.seed(42)\n",
    "print(f'pandas {pd.__version__} | numpy {np.__version__} | Ready.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── PROJECT PATHS — nothing hardcoded ────────────────────────────────────────\n",
    "# Override DATA_DIR via environment variable or set it directly below.\n",
    "#\n",
    "# Google Colab:  from google.colab import drive; drive.mount('/content/drive')\n",
    "#                DATA_DIR = Path('/content/drive/MyDrive/chaguoai/data')\n",
    "# Kaggle:        DATA_DIR = Path('/kaggle/input/chaguoai-data')\n",
    "# Local:         DATA_DIR = Path('data/raw')\n",
    "# ─────────────────────────────────────────────────────────────────────────────\n",
    "\n",
    "DATA_DIR    = Path(os.getenv('CHAGUOAI_DATA_DIR', 'data/raw'))\n",
    "OUTPUTS_DIR = Path(os.getenv('CHAGUOAI_OUTPUTS_DIR', 'outputs'))\n",
    "\n",
    "FIGURES_DIR = OUTPUTS_DIR / 'figures'\n",
    "MODELS_DIR  = OUTPUTS_DIR / 'models'\n",
    "REPORTS_DIR = OUTPUTS_DIR / 'reports'\n",
    "\n",
    "for d in [FIGURES_DIR, MODELS_DIR, REPORTS_DIR]:\n",
    "    d.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "CSS_PATH = DATA_DIR / 'Client_Service_Statistics.csv'\n",
    "WOM_PATH = DATA_DIR / 'Final_women_Data_ANON.csv'\n",
    "\n",
    "RANDOM_SEED = 42\n",
    "TRAIN_RATIO = 0.70\n",
    "VAL_RATIO   = 0.15\n",
    "TEST_RATIO  = 0.15\n",
    "MIN_AUC     = 0.65\n",
    "MIN_RECALL  = 0.50\n",
    "\n",
    "print(f'Data dir:    {DATA_DIR.resolve()}')\n",
    "print(f'Outputs dir: {OUTPUTS_DIR.resolve()}')\n",
    "print(f'CSS exists: {CSS_PATH.exists()} | WOM exists: {WOM_PATH.exists()}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Data Loading and Validation"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_csv_safe(path: Path, **kwargs) -> pd.DataFrame:\n",
    "    \"\"\"Load CSV with automatic encoding detection. Fails clearly if file missing.\"\"\"\n",
    "    if not path.exists():\n",
    "        raise FileNotFoundError(\n",
    "            f'File not found: {path}\\n'\n",
    "            f'Set DATA_DIR to the folder containing your CSV files.'\n",
    "        )\n",
    "    for enc in ['utf-8', 'latin1', 'cp1252', 'utf-8-sig']:\n",
    "        try:\n",
    "            df = pd.read_csv(path, encoding=enc, low_memory=False, **kwargs)\n",
    "            print(f'  {path.name} [{enc}] — {df.shape[0]:,} rows x {df.shape[1]} cols')\n",
    "            return df\n",
    "        except (UnicodeDecodeError, UnicodeError):\n",
    "            continue\n",
    "    raise ValueError(f'Cannot decode {path} with any standard encoding.')\n",
    "\n",
    "\n",
    "def validate_dataset(df: pd.DataFrame, name: str, required_cols: list) -> dict:\n",
    "    \"\"\"Basic data quality checks. Returns dict of issues and warnings.\"\"\"\n",
    "    issues, warnings_list = [], []\n",
    "    missing_cols = [c for c in required_cols if c not in df.columns]\n",
    "    if missing_cols:\n",
    "        issues.append(f'Missing columns: {missing_cols}')\n",
    "    n_dupes = df.duplicated().sum()\n",
    "    if n_dupes > 0:\n",
    "        warnings_list.append(f'{n_dupes:,} duplicate rows')\n",
    "    high_miss = df.isnull().mean()\n",
    "    high_miss = high_miss[high_miss > 0.80]\n",
    "    if len(high_miss):\n",
    "        warnings_list.append(f'{len(high_miss)} columns with >80% missing')\n",
    "    return {'name': name, 'issues': issues, 'warnings': warnings_list,\n",
    "            'n_rows': len(df), 'n_cols': len(df.columns),\n",
    "            'overall_missing_pct': round(df.isnull().mean().mean()*100, 1)}\n",
    "\n",
    "\n",
    "print('Loading datasets...')\n",
    "css_raw = load_csv_safe(CSS_PATH)\n",
    "wom_raw = load_csv_safe(WOM_PATH)\n",
    "\n",
    "CSS_REQUIRED = ['age','educationlevel','noofchildren','fertilityintention',\n",
    "                'fpstatus','previousmethod','methodadopted','counseled','gender']\n",
    "WOM_REQUIRED = ['q102','q105','q203_son','q203_daug','q310','q311','q334a','district']\n",
    "\n",
    "for df, name, required in [(css_raw,'CSS',CSS_REQUIRED),(wom_raw,'Women Survey',WOM_REQUIRED)]:\n",
    "    r = validate_dataset(df, name, required)\n",
    "    print(f\"\\n{r['name']}: {r['n_rows']:,} rows | {r['overall_missing_pct']}% missing overall\")\n",
    "    for iss in r['issues']:   print(f'  ISSUE:   {iss}')\n",
    "    for wrn in r['warnings']: print(f'  WARNING: {wrn}')\n",
    "    if not r['issues'] and not r['warnings']: print('  All checks passed')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Exploratory Data Analysis"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Chart style\n",
    "PALETTE = {'teal':'#2A9D8F','coral':'#E76F51','amber':'#E9C46A',\n",
    "           'navy':'#264653','sage':'#52796F','red':'#C62828','gray':'#9E9E9E'}\n",
    "\n",
    "plt.rcParams.update({\n",
    "    'figure.facecolor':'white','axes.facecolor':'white',\n",
    "    'axes.spines.top':False,'axes.spines.right':False,\n",
    "    'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold','figure.dpi':120\n",
    "})\n",
    "\n",
    "def save_fig(name):\n",
    "    plt.savefig(FIGURES_DIR/f'{name}.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()\n",
    "    print(f'  Saved: {name}.png')\n",
    "\n",
    "# ── Prepare CSS female-revisit subset for EDA ─────────────────────────────────\n",
    "css_female   = css_raw[css_raw['gender']=='Female'].copy()\n",
    "css_revisit  = css_female[\n",
    "    (css_female['fpstatus']=='Revisit') &\n",
    "    css_female['previousmethod'].notna() &\n",
    "    (css_female['previousmethod']!='Missing')\n",
    "].copy()\n",
    "\n",
    "css_revisit['discontinued'] = (\n",
    "    css_revisit['previousmethod'].str.strip().str.lower() !=\n",
    "    css_revisit['methodadopted'].str.strip().str.lower()\n",
    ").astype(int)\n",
    "\n",
    "overall_disc_rate = css_revisit['discontinued'].mean()\n",
    "\n",
    "print(f'Female records:        {len(css_female):,}')\n",
    "print(f'Female revisits usable:{len(css_revisit):,}')\n",
    "print(f'Overall switch rate:   {overall_disc_rate:.1%}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 1: Dataset overview ─────────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n",
    "\n",
    "ax = axes[0]\n",
    "bars_data = {\n",
    "    'CSS\\nAll records':        len(css_raw),\n",
    "    'CSS\\nFemale':             len(css_female),\n",
    "    'CSS\\nRevisits\\n(model)':  len(css_revisit),\n",
    "    'Women\\nSurvey':           len(wom_raw),\n",
    "}\n",
    "colors = [PALETTE['navy'],PALETTE['teal'],PALETTE['coral'],PALETTE['amber']]\n",
    "bars = ax.barh(list(bars_data.keys())[::-1], list(bars_data.values())[::-1], color=colors[::-1], height=0.5)\n",
    "for bar, val in zip(bars, list(bars_data.values())[::-1]):\n",
    "    ax.text(bar.get_width()+500, bar.get_y()+bar.get_height()/2, f'{val:,}', va='center', fontsize=9)\n",
    "ax.set_title('Record Counts by Dataset')\n",
    "ax.set_xlabel('Records')\n",
    "\n",
    "ax = axes[1]\n",
    "year_counts = css_raw['year'].value_counts().sort_index()\n",
    "ax.bar(year_counts.index.astype(str), year_counts.values, color=PALETTE['teal'], width=0.5)\n",
    "for i, v in enumerate(year_counts.values):\n",
    "    ax.text(i, v+500, f'{v:,}', ha='center', fontsize=9)\n",
    "ax.set_title('CSS Visits by Year')\n",
    "ax.set_xlabel('Year')\n",
    "ax.set_ylabel('Visits')\n",
    "\n",
    "ax = axes[2]\n",
    "county_counts = css_female['county'].value_counts()\n",
    "ax.pie(county_counts.values, labels=county_counts.index,\n",
    "       colors=[PALETTE['teal'], PALETTE['coral']], autopct='%1.0f%%', startangle=90)\n",
    "ax.set_title('Female Clients by County')\n",
    "\n",
    "plt.suptitle('Chart 1: Dataset Overview', fontweight='bold', y=1.01)\n",
    "plt.tight_layout()\n",
    "save_fig('01_dataset_overview')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 2: Age distributions ─────────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 2, figsize=(13, 4))\n",
    "\n",
    "css_age = pd.to_numeric(css_female['age'], errors='coerce').dropna()\n",
    "css_age = css_age[(css_age>=10)&(css_age<=60)]\n",
    "wom_age = pd.to_numeric(wom_raw['q102'], errors='coerce').dropna()\n",
    "wom_age = wom_age[(wom_age>=10)&(wom_age<=60)]\n",
    "\n",
    "for ax, data, title, color in [\n",
    "    (axes[0], css_age, f'CSS Female (n={len(css_female):,})', PALETTE['teal']),\n",
    "    (axes[1], wom_age, f'Women Survey (n={len(wom_raw):,})', PALETTE['coral'])\n",
    "]:\n",
    "    ax.hist(data, bins=range(10,62,2), color=color, edgecolor='white', alpha=0.9)\n",
    "    ax.axvline(data.median(), color=PALETTE['red'], linewidth=2, linestyle='--',\n",
    "               label=f'Median: {data.median():.0f}')\n",
    "    ax.set_title(title)\n",
    "    ax.set_xlabel('Age (years)')\n",
    "    ax.set_ylabel('Count')\n",
    "    ax.legend()\n",
    "\n",
    "plt.suptitle('Chart 2: Age Distribution', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "save_fig('02_age_distribution')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 3: Method mix ────────────────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "ax = axes[0]\n",
    "method_counts = css_female['methodadopted'].str.strip().value_counts().head(10).sort_values()\n",
    "method_counts.plot.barh(ax=ax, color=PALETTE['teal'])\n",
    "for i, v in enumerate(method_counts.values):\n",
    "    ax.text(v+100, i, f'{v:,}', va='center', fontsize=9)\n",
    "ax.set_title('Methods Adopted (CSS Female)')\n",
    "ax.set_xlabel('Clients')\n",
    "\n",
    "ax = axes[1]\n",
    "wom_methods = wom_raw['q311'].dropna().str.strip().value_counts().head(10).sort_values()\n",
    "wom_methods.plot.barh(ax=ax, color=PALETTE['coral'])\n",
    "for i, v in enumerate(wom_methods.values):\n",
    "    ax.text(v+1, i, f'{v}', va='center', fontsize=9)\n",
    "ax.set_title('Current Method (Women Survey)')\n",
    "ax.set_xlabel('Women')\n",
    "\n",
    "plt.suptitle('Chart 3: Contraceptive Method Mix', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "save_fig('03_method_mix')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 4: Discontinuation rate by method ───────────────────────────────────\n",
    "disc_by_method = (\n",
    "    css_revisit.groupby('previousmethod')['discontinued']\n",
    "    .agg(['mean','count']).reset_index()\n",
    "    .rename(columns={'mean':'disc_rate','count':'n'})\n",
    "    .query('n >= 200')\n",
    "    .sort_values('disc_rate')\n",
    ")\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "colors = [PALETTE['coral'] if r >= 0.5 else PALETTE['teal'] for r in disc_by_method['disc_rate']]\n",
    "bars = ax.barh(disc_by_method['previousmethod'], disc_by_method['disc_rate']*100, color=colors, height=0.6)\n",
    "for bar, (_, row) in zip(bars, disc_by_method.iterrows()):\n",
    "    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,\n",
    "            f\"{row['disc_rate']*100:.1f}%  (n={row['n']:,})\", va='center', fontsize=9.5)\n",
    "ax.axvline(overall_disc_rate*100, color=PALETTE['red'], linestyle='--', linewidth=1.5,\n",
    "           label=f'Overall: {overall_disc_rate*100:.1f}%')\n",
    "ax.set_xlabel('Switch rate (%)')\n",
    "ax.set_title('Chart 4: Method Switch Rate by Previous Method\\n(CSS Revisit Clients)')\n",
    "ax.legend()\n",
    "ax.set_xlim(0, 110)\n",
    "plt.tight_layout()\n",
    "save_fig('04_disc_by_method')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Charts 5 & 6: By age group and parity ─────────────────────────────────────\n",
    "css_revisit['age_num'] = pd.to_numeric(css_revisit['age'], errors='coerce')\n",
    "css_revisit['age_group'] = pd.cut(\n",
    "    css_revisit['age_num'], bins=[9,17,24,29,34,39,49,60],\n",
    "    labels=['10-17','18-24','25-29','30-34','35-39','40-49','50+']\n",
    ")\n",
    "css_revisit['parity_group'] = pd.cut(\n",
    "    pd.to_numeric(css_revisit['noofchildren'],errors='coerce'),\n",
    "    bins=[-1,0,1,2,4,20],\n",
    "    labels=['0 children','1 child','2 children','3-4 children','5+ children']\n",
    ")\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))\n",
    "\n",
    "for ax, group_col, title, color in [\n",
    "    (axes[0], 'age_group',   'Chart 5: Switch Rate by Age Group',          PALETTE['teal']),\n",
    "    (axes[1], 'parity_group','Chart 6: Switch Rate by Number of Children', PALETTE['coral']),\n",
    "]:\n",
    "    data = (\n",
    "        css_revisit.groupby(group_col, observed=True)['discontinued']\n",
    "        .agg(['mean','count']).reset_index().query('count>=50')\n",
    "    )\n",
    "    bars = ax.bar(data[group_col].astype(str), data['mean']*100, color=color, width=0.6)\n",
    "    for bar, (_, row) in zip(bars, data.iterrows()):\n",
    "        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,\n",
    "                f\"{row['mean']*100:.0f}%\", ha='center', va='bottom', fontsize=8.5)\n",
    "    ax.axhline(overall_disc_rate*100, color=PALETTE['red'], linestyle='--',\n",
    "               linewidth=1.5, label=f'Overall: {overall_disc_rate*100:.1f}%')\n",
    "    ax.set_title(title)\n",
    "    ax.set_ylabel('Switch rate (%)')\n",
    "    ax.legend(fontsize=8)\n",
    "    plt.setp(ax.get_xticklabels(), rotation=15, ha='right')\n",
    "\n",
    "plt.tight_layout()\n",
    "save_fig('05_disc_age_parity')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 7: Reasons women give for not using FP (Women Survey) ────────────────\n",
    "reasons = (\n",
    "    wom_raw['q341_1st'].dropna().str.strip()\n",
    "    .replace({'99':None}).dropna().value_counts().head(12).sort_values()\n",
    ")\n",
    "\n",
    "reason_color_map = {\n",
    "    'Fear Of Side Effects':      PALETTE['red'],\n",
    "    'Health Concerns':           PALETTE['red'],\n",
    "    'Others Oppose':             PALETTE['coral'],\n",
    "    'Partner Opposes':           PALETTE['coral'],\n",
    "    'Infrequent Sex/No Sex':     PALETTE['amber'],\n",
    "    'Already Pregnant':          PALETTE['amber'],\n",
    "    'Breastfeeding':             PALETTE['teal'],\n",
    "    'Wants More Children/Trying To Get Pregnant': PALETTE['teal'],\n",
    "}\n",
    "colors = [reason_color_map.get(r, PALETTE['gray']) for r in reasons.index]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(11, 7))\n",
    "reasons.plot.barh(ax=ax, color=colors)\n",
    "for i, v in enumerate(reasons.values):\n",
    "    ax.text(v+1, i, f'{v}', va='center', fontsize=9)\n",
    "\n",
    "legend_items = [\n",
    "    mpatches.Patch(color=PALETTE['red'],   label='Health / Side effects'),\n",
    "    mpatches.Patch(color=PALETTE['coral'], label='Social / Partner opposition'),\n",
    "    mpatches.Patch(color=PALETTE['amber'], label='Behavioural / Life stage'),\n",
    "    mpatches.Patch(color=PALETTE['teal'],  label='Fertility-related'),\n",
    "    mpatches.Patch(color=PALETTE['gray'],  label='Other'),\n",
    "]\n",
    "ax.legend(handles=legend_items, loc='lower right', fontsize=9)\n",
    "ax.set_xlabel('Number of women')\n",
    "ax.set_title('Chart 7: Why Women Are Not Using FP\\n(Women Survey — q341_1st first-mention reason)')\n",
    "plt.tight_layout()\n",
    "save_fig('06_nonuse_reasons')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 8: Method switch heatmap ─────────────────────────────────────────────\n",
    "switchers = css_revisit[css_revisit['discontinued']==1]\n",
    "switch_matrix = switchers.groupby(['previousmethod','methodadopted']).size().unstack(fill_value=0)\n",
    "\n",
    "top_prev    = css_revisit['previousmethod'].value_counts().head(5).index\n",
    "top_adopted = css_female['methodadopted'].str.strip().value_counts().head(7).index\n",
    "switch_sub  = switch_matrix.reindex(\n",
    "    [m for m in top_prev    if m in switch_matrix.index]\n",
    ")[[c for c in top_adopted if c in switch_matrix.columns]].fillna(0)\n",
    "\n",
    "if not switch_sub.empty:\n",
    "    fig, ax = plt.subplots(figsize=(12, 5))\n",
    "    sns.heatmap(switch_sub, annot=True, fmt=',d', cmap='YlOrRd', linewidths=0.5, ax=ax)\n",
    "    ax.set_ylabel('Previous method (FROM)')\n",
    "    ax.set_xlabel('Method adopted (TO)')\n",
    "    ax.set_title('Chart 8: Method Switch Pattern\\n(Off-diagonal = discontinued and switched)')\n",
    "    plt.xticks(rotation=30, ha='right')\n",
    "    plt.tight_layout()\n",
    "    save_fig('07_switch_heatmap')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 9: Fertility intention vs method ─────────────────────────────────────\n",
    "df_plot = css_revisit[\n",
    "    css_revisit['fertilityintention'].isin(['Within 2 Years','Later than 2 years','No more Children'])\n",
    "].copy()\n",
    "df_plot['prev_cat'] = df_plot['previousmethod'].map({\n",
    "    'Injectables':'Short-acting','Pills':'Short-acting','Condom':'Barrier',\n",
    "    'Implants':'Long-acting','IUCD':'Long-acting','BTL':'Permanent'\n",
    "}).fillna('Other')\n",
    "\n",
    "pivot = (\n",
    "    df_plot.groupby(['fertilityintention','prev_cat']).size()\n",
    "    .unstack(fill_value=0)\n",
    "    .apply(lambda row: row/row.sum()*100, axis=1)\n",
    ")\n",
    "cat_colors = {'Short-acting':PALETTE['coral'],'Long-acting':PALETTE['teal'],\n",
    "              'Barrier':PALETTE['amber'],'Permanent':PALETTE['navy'],'Other':PALETTE['gray']}\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(11, 5))\n",
    "pivot.plot.bar(ax=ax, stacked=True,\n",
    "               color=[cat_colors.get(c, PALETTE['gray']) for c in pivot.columns])\n",
    "ax.set_title('Chart 9: Previous Method Category by Fertility Intention')\n",
    "ax.set_xlabel('Fertility intention')\n",
    "ax.set_ylabel('% of clients')\n",
    "ax.legend(title='Method category', bbox_to_anchor=(1.01, 1))\n",
    "plt.xticks(rotation=10)\n",
    "plt.tight_layout()\n",
    "save_fig('08_fertility_method')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Chart 10: Missing data rates ───────────────────────────────────────────────\n",
    "model_cols = ['age','educationlevel','noofchildren','fertilityintention',\n",
    "              'previousmethod','methodadopted','counseled','county']\n",
    "miss_df = pd.DataFrame({\n",
    "    'column': model_cols,\n",
    "    'missing_pct': [css_revisit[c].isna().mean()*100\n",
    "                    if c in css_revisit.columns else 100 for c in model_cols]\n",
    "}).sort_values('missing_pct')\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(9, 4))\n",
    "colors = [PALETTE['red'] if v>40 else PALETTE['amber'] if v>15 else PALETTE['teal']\n",
    "          for v in miss_df['missing_pct']]\n",
    "ax.barh(miss_df['column'], miss_df['missing_pct'], color=colors, height=0.5)\n",
    "for i, (_, row) in enumerate(miss_df.iterrows()):\n",
    "    ax.text(row['missing_pct']+0.3, i, f\"{row['missing_pct']:.1f}%\", va='center', fontsize=9)\n",
    "ax.axvline(30, color=PALETTE['gray'], linestyle='--', label='30% threshold')\n",
    "ax.set_title('Chart 10: Missing Data — Model Feature Columns')\n",
    "ax.set_xlabel('Missing (%)')\n",
    "ax.set_xlim(0, 105)\n",
    "ax.legend()\n",
    "plt.tight_layout()\n",
    "save_fig('09_missing_data')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Charts 11 & 12: Counseling impact and county comparison ───────────────────\n",
    "fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))\n",
    "\n",
    "# Counseling vs discontinuation\n",
    "ax = axes[0]\n",
    "counsel_disc = (\n",
    "    css_revisit[css_revisit['counseled'].isin(['Yes','No'])]\n",
    "    .groupby('counseled')['discontinued']\n",
    "    .agg(['mean','count']).reset_index()\n",
    ")\n",
    "bars = ax.bar(counsel_disc['counseled'], counsel_disc['mean']*100,\n",
    "              color=[PALETTE['teal'],PALETTE['coral']], width=0.4)\n",
    "for bar, (_, row) in zip(bars, counsel_disc.iterrows()):\n",
    "    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,\n",
    "            f\"{row['mean']*100:.1f}%\\nn={row['count']:,}\", ha='center', va='bottom')\n",
    "ax.set_title('Chart 11: Switch Rate by Counseling Status')\n",
    "ax.set_ylabel('Switch rate (%)')\n",
    "ax.set_xlabel('Was client counseled?')\n",
    "\n",
    "# County comparison\n",
    "ax = axes[1]\n",
    "county_disc = (\n",
    "    css_revisit.groupby('county')['discontinued']\n",
    "    .agg(['mean','count']).reset_index()\n",
    ")\n",
    "bars = ax.bar(county_disc['county'], county_disc['mean']*100,\n",
    "              color=[PALETTE['teal'],PALETTE['coral']], width=0.4)\n",
    "for bar, (_, row) in zip(bars, county_disc.iterrows()):\n",
    "    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,\n",
    "            f\"{row['mean']*100:.1f}%\\nn={row['count']:,}\", ha='center', va='bottom')\n",
    "ax.set_title('Chart 12: Switch Rate by County')\n",
    "ax.set_ylabel('Switch rate (%)')\n",
    "ax.set_xlabel('County')\n",
    "\n",
    "plt.tight_layout()\n",
    "save_fig('10_counseling_county')\n",
    "\n",
    "print('All 12 EDA charts complete.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Feature Engineering"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Method → clinical category mapping\n",
    "METHOD_CATEGORY_MAP = {\n",
    "    'Implants':       'long_acting_reversible',\n",
    "    'IUCD':           'long_acting_reversible',\n",
    "    'Pills':          'short_acting_hormonal',\n",
    "    'Injectables':    'short_acting_hormonal',\n",
    "    'Pills & Condoms':'short_acting_hormonal',\n",
    "    'Condoms':        'barrier',\n",
    "    'BTL':            'permanent',\n",
    "    'NSV':            'permanent',\n",
    "    'Male condom':    'barrier',\n",
    "}\n",
    "\n",
    "EDUCATION_MAP = {'Primary Incomplete':1,'Primary Complete':2,'Secondary & Above':3,'Missing':None}\n",
    "FERTILITY_MAP = {'Within 2 Years':1,'Later than 2 years':2,'No more Children':3,'Missing':None}\n",
    "COUNSELED_MAP = {'Yes':1,'Refreshers':1,'No':0,'Missing':0}\n",
    "\n",
    "def categorize_method(method_name):\n",
    "    if pd.isna(method_name): return 'unknown'\n",
    "    return METHOD_CATEGORY_MAP.get(str(method_name).strip(), 'other')\n",
    "\n",
    "def switch_direction(curr_cat, prev_cat):\n",
    "    if prev_cat in ('unknown','other'): return 'unknown'\n",
    "    if curr_cat == prev_cat: return 'same'\n",
    "    if prev_cat=='long_acting_reversible' and curr_cat!='long_acting_reversible': return 'downgrade'\n",
    "    if curr_cat=='long_acting_reversible' and prev_cat!='long_acting_reversible': return 'upgrade'\n",
    "    if curr_cat=='permanent': return 'to_permanent'\n",
    "    return 'lateral'\n",
    "\n",
    "\n",
    "def build_features(df: pd.DataFrame, dataset_name: str = 'unknown') -> pd.DataFrame:\n",
    "    \"\"\"\n",
    "    Transform raw CSS revisit data into model-ready features.\n",
    "    PORTABLE: depends only on input DataFrame and module-level mappings.\n",
    "    Safe to call on new CSV data from any county or country.\n",
    "    \"\"\"\n",
    "    out = pd.DataFrame(index=df.index)\n",
    "    out['dataset_source'] = dataset_name\n",
    "\n",
    "    # Age\n",
    "    out['age_years'] = pd.to_numeric(df['age'], errors='coerce').clip(10,60)\n",
    "    age_med = out['age_years'].median()\n",
    "    out['age_years'] = out['age_years'].fillna(age_med)\n",
    "    out['is_young_woman'] = (out['age_years'] < 20).astype(int)\n",
    "    out['is_older_woman'] = (out['age_years'] >= 40).astype(int)\n",
    "\n",
    "    # Parity\n",
    "    out['number_of_children'] = pd.to_numeric(df['noofchildren'], errors='coerce').clip(0,15)\n",
    "    out['number_of_children'] = out['number_of_children'].fillna(out['number_of_children'].median())\n",
    "    out['has_high_parity']   = (out['number_of_children'] >= 5).astype(int)\n",
    "    out['is_nulliparous']    = (out['number_of_children'] == 0).astype(int)\n",
    "\n",
    "    # Education\n",
    "    out['education_level_numeric'] = df['educationlevel'].map(EDUCATION_MAP)\n",
    "    edu_med = out['education_level_numeric'].median()\n",
    "    out['education_level_numeric'] = out['education_level_numeric'].fillna(\n",
    "        edu_med if pd.notna(edu_med) else 2\n",
    "    )\n",
    "\n",
    "    # Fertility intention\n",
    "    out['fertility_intention_encoded'] = df['fertilityintention'].map(FERTILITY_MAP)\n",
    "    out['fertility_intention_encoded'] = out['fertility_intention_encoded'].fillna(\n",
    "        out['fertility_intention_encoded'].median()\n",
    "    ).fillna(2)\n",
    "    out['wants_child_soon']       = (df['fertilityintention']=='Within 2 Years').astype(int)\n",
    "    out['wants_no_more_children'] = (df['fertilityintention']=='No more Children').astype(int)\n",
    "\n",
    "    # Methods\n",
    "    out['previous_method_category'] = df['previousmethod'].apply(categorize_method)\n",
    "    out['current_method_category']  = df['methodadopted'].apply(categorize_method)\n",
    "    out['adopted_long_acting']      = (out['current_method_category']=='long_acting_reversible').astype(int)\n",
    "    out['was_on_long_acting']       = (out['previous_method_category']=='long_acting_reversible').astype(int)\n",
    "    out['switch_direction'] = out.apply(\n",
    "        lambda r: switch_direction(r['current_method_category'], r['previous_method_category']), axis=1\n",
    "    )\n",
    "\n",
    "    # Counseling\n",
    "    out['was_counseled_binary'] = df['counseled'].map(COUNSELED_MAP).fillna(0).astype(int)\n",
    "\n",
    "    # Geography\n",
    "    out['county'] = df['county'].str.strip().fillna('Unknown') if 'county' in df.columns else 'Unknown'\n",
    "\n",
    "    # Temporal context (for diagnostics, not in model)\n",
    "    if 'year' in df.columns:\n",
    "        out['service_year'] = pd.to_numeric(df['year'], errors='coerce')\n",
    "\n",
    "    return out\n",
    "\n",
    "\n",
    "print('Building feature matrix...')\n",
    "features_raw = build_features(css_revisit, dataset_name='css_western_kenya_2013_2015')\n",
    "print(f'Feature matrix: {features_raw.shape}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Target Variable"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Attach target — exclude planned removal visits (not unplanned discontinuation)\n",
    "REMOVAL_METHODS = ['Removal_Implant','Removal_IUCD','Removal_Implants']\n",
    "is_removal = css_revisit['methodadopted'].str.strip().isin(REMOVAL_METHODS)\n",
    "\n",
    "features_raw['discontinued'] = css_revisit['discontinued'].values\n",
    "features_raw = features_raw[~is_removal].copy()\n",
    "\n",
    "minority_rate = features_raw['discontinued'].mean()\n",
    "print(f'Records after excluding removals: {len(features_raw):,}')\n",
    "print(f'Continued    (0): {(features_raw[\"discontinued\"]==0).sum():,} ({1-minority_rate:.1%})')\n",
    "print(f'Discontinued (1): {(features_raw[\"discontinued\"]==1).sum():,} ({minority_rate:.1%})')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Encoding and Final Feature Matrix"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "CATEGORICAL_FEATURES = ['previous_method_category','current_method_category',\n",
    "                         'switch_direction','county','dataset_source']\n",
    "\n",
    "NUMERIC_FEATURES = [\n",
    "    'age_years','number_of_children','education_level_numeric',\n",
    "    'fertility_intention_encoded','is_young_woman','is_older_woman',\n",
    "    'has_high_parity','is_nulliparous','adopted_long_acting','was_on_long_acting',\n",
    "    'wants_child_soon','wants_no_more_children','was_counseled_binary',\n",
    "]\n",
    "\n",
    "\n",
    "def encode_categoricals(df, categorical_cols, fit=True, encoders=None):\n",
    "    \"\"\"\n",
    "    Encode categorical columns with LabelEncoder.\n",
    "    fit=True  → fit new encoders (training data only).\n",
    "    fit=False → transform using existing encoders (val/test/inference).\n",
    "    Unknown values at inference time are mapped to 'unknown' class.\n",
    "    \"\"\"\n",
    "    if fit: encoders = {}\n",
    "    df_out = df.copy()\n",
    "    for col in categorical_cols:\n",
    "        series = df_out[col].fillna('unknown').astype(str)\n",
    "        if fit:\n",
    "            le = LabelEncoder()\n",
    "            le.fit(series.unique().tolist() + ['unknown'])\n",
    "            encoders[col] = le\n",
    "        else:\n",
    "            le = encoders[col]\n",
    "            known = set(le.classes_)\n",
    "            series = series.apply(lambda v: v if v in known else 'unknown')\n",
    "        df_out[col+'_encoded'] = le.transform(series)\n",
    "    return df_out, encoders\n",
    "\n",
    "\n",
    "features_encoded, encoders = encode_categoricals(features_raw, CATEGORICAL_FEATURES, fit=True)\n",
    "\n",
    "feature_col_names = NUMERIC_FEATURES + [c+'_encoded' for c in CATEGORICAL_FEATURES]\n",
    "available_features = [c for c in feature_col_names if c in features_encoded.columns]\n",
    "missing_features   = [c for c in feature_col_names if c not in features_encoded.columns]\n",
    "\n",
    "if missing_features: print(f'WARNING: Missing features: {missing_features}')\n",
    "\n",
    "X = features_encoded[available_features].copy()\n",
    "y = features_encoded['discontinued'].astype(int)\n",
    "\n",
    "# Final NaN check — never allow silent NaN propagation into model\n",
    "nan_counts = X.isna().sum()\n",
    "if nan_counts.any():\n",
    "    print(f'Remaining NaNs (applying median imputation):')\n",
    "    print(nan_counts[nan_counts>0])\n",
    "    X = X.fillna(X.median())\n",
    "\n",
    "assert X.isna().sum().sum() == 0, 'NaN values still present after imputation.'\n",
    "\n",
    "print(f'\\nFinal X: {X.shape}')\n",
    "print(f'Target positive rate: {y.mean():.1%}')\n",
    "print(f'Features: {list(X.columns)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 7. 70 / 15 / 15 Data Split"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Step 1: carve out 15% test — sealed, not touched until final evaluation\n",
    "X_temp, X_test, y_temp, y_test = train_test_split(\n",
    "    X, y, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y\n",
    ")\n",
    "\n",
    "# Step 2: split remaining 85% into 70% train and 15% val\n",
    "# Val fraction of remaining = 0.15 / 0.85 = 0.1765\n",
    "val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)\n",
    "X_train, X_val, y_train, y_val = train_test_split(\n",
    "    X_temp, y_temp, test_size=val_frac, random_state=RANDOM_SEED, stratify=y_temp\n",
    ")\n",
    "\n",
    "print('Split complete (all stratified on target):')\n",
    "for split_name, Xs, ys in [('Train',X_train,y_train),('Val',X_val,y_val),('Test',X_test,y_test)]:\n",
    "    print(f'  {split_name:<6}: {len(Xs):,} rows ({len(Xs)/len(X)*100:.0f}%) | pos rate: {ys.mean():.1%}')\n",
    "print('\\nTest set is now sealed.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 8. Class Imbalance"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "minority_train = y_train.mean()\n",
    "print(f'Training positive rate: {minority_train:.1%}')\n",
    "\n",
    "if minority_train >= 0.30:\n",
    "    balance_strategy = 'none'\n",
    "    use_class_weight = None\n",
    "    print('Good balance — no special handling.')\n",
    "elif minority_train >= 0.15:\n",
    "    balance_strategy = 'class_weight'\n",
    "    use_class_weight = 'balanced'\n",
    "    print('Moderate imbalance — using class_weight=balanced.')\n",
    "else:\n",
    "    balance_strategy = 'smote'\n",
    "    use_class_weight = 'balanced'\n",
    "    print('Severe imbalance — applying SMOTE to training set only.')\n",
    "\n",
    "X_train_bal, y_train_bal = X_train.copy(), y_train.copy()\n",
    "\n",
    "if balance_strategy == 'smote':\n",
    "    smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)\n",
    "    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)\n",
    "    print(f'After SMOTE: {len(X_train_bal):,} rows | pos rate: {y_train_bal.mean():.1%}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 9. Model Training and Cross-Validation"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)\n",
    "\n",
    "candidate_models = {\n",
    "    'logistic_regression': LogisticRegression(\n",
    "        class_weight=use_class_weight, max_iter=1000,\n",
    "        random_state=RANDOM_SEED, solver='lbfgs'\n",
    "    ),\n",
    "    'random_forest': RandomForestClassifier(\n",
    "        n_estimators=300, max_depth=8, min_samples_leaf=5,\n",
    "        class_weight=use_class_weight, random_state=RANDOM_SEED, n_jobs=-1\n",
    "    ),\n",
    "    'xgboost': xgb.XGBClassifier(\n",
    "        n_estimators=300, max_depth=5, learning_rate=0.05,\n",
    "        subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',\n",
    "        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0\n",
    "    ),\n",
    "    'lightgbm': lgb.LGBMClassifier(\n",
    "        n_estimators=300, max_depth=5, learning_rate=0.05,\n",
    "        class_weight=use_class_weight, random_state=RANDOM_SEED,\n",
    "        n_jobs=-1, verbose=-1\n",
    "    ),\n",
    "}\n",
    "\n",
    "print(f\"{'Model':<25} {'CV AUC':>9} {'CV Std':>8} {'Val AUC':>9} {'Val AP':>8}\")\n",
    "print('-' * 63)\n",
    "\n",
    "comparison = {}\n",
    "for name, model in candidate_models.items():\n",
    "    cv_auc = cross_val_score(model, X_train_bal, y_train_bal,\n",
    "                              cv=cv, scoring='roc_auc', n_jobs=-1)\n",
    "    model.fit(X_train_bal, y_train_bal)\n",
    "    y_val_prob = model.predict_proba(X_val)[:,1]\n",
    "    val_auc    = roc_auc_score(y_val, y_val_prob)\n",
    "    val_ap     = average_precision_score(y_val, y_val_prob)\n",
    "    comparison[name] = dict(cv_mean=cv_auc.mean(), cv_std=cv_auc.std(),\n",
    "                             val_auc=val_auc, val_ap=val_ap, model=model)\n",
    "    print(f\"{name:<25} {cv_auc.mean():>9.3f} {cv_auc.std():>8.3f} {val_auc:>9.3f} {val_ap:>8.3f}\")\n",
    "\n",
    "best_name  = max(comparison, key=lambda k: comparison[k]['val_auc'])\n",
    "best_model = comparison[best_name]['model']\n",
    "print(f'\\nBest model: {best_name} | Val AUC: {comparison[best_name][\"val_auc\"]:.4f}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Model comparison chart\n",
    "names    = list(comparison.keys())\n",
    "val_aucs = [comparison[m]['val_auc'] for m in names]\n",
    "cv_aucs  = [comparison[m]['cv_mean'] for m in names]\n",
    "cv_stds  = [comparison[m]['cv_std']  for m in names]\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(13, 4))\n",
    "\n",
    "ax = axes[0]\n",
    "bars = ax.barh(names, val_aucs,\n",
    "               color=[PALETTE['coral'] if m==best_name else PALETTE['teal'] for m in names],\n",
    "               height=0.5)\n",
    "ax.axvline(MIN_AUC, color=PALETTE['red'], linestyle='--', label=f'Min acceptable: {MIN_AUC}')\n",
    "ax.axvline(0.5,     color=PALETTE['gray'], linestyle=':', label='Random: 0.50')\n",
    "for bar, val in zip(bars, val_aucs):\n",
    "    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,\n",
    "            f'{val:.3f}', va='center', fontsize=9.5)\n",
    "ax.set_title('Validation Set AUC-ROC')\n",
    "ax.set_xlim(0.4, 1.0)\n",
    "ax.legend(fontsize=8)\n",
    "\n",
    "ax = axes[1]\n",
    "xp = range(len(names))\n",
    "ax.bar(xp, cv_aucs, color=PALETTE['teal'], width=0.5, yerr=cv_stds, capsize=5)\n",
    "ax.set_xticks(xp)\n",
    "ax.set_xticklabels([n.replace('_','\\n') for n in names], fontsize=9)\n",
    "ax.axhline(MIN_AUC, color=PALETTE['red'], linestyle='--')\n",
    "ax.set_title('Cross-Validation AUC (5-fold ± std)')\n",
    "ax.set_ylabel('AUC-ROC')\n",
    "ax.set_ylim(0.4, 1.0)\n",
    "\n",
    "plt.suptitle('Model Comparison Chart', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "save_fig('11_model_comparison')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 10. Threshold Optimisation on Validation Set"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Tune decision threshold on VALIDATION set only — never on test\n",
    "y_val_prob = best_model.predict_proba(X_val)[:,1]\n",
    "precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)\n",
    "f1_scores = [2*p*r/(p+r) if (p+r)>0 else 0\n",
    "             for p, r in zip(precisions[:-1], recalls[:-1])]\n",
    "best_idx   = int(np.argmax(f1_scores))\n",
    "opt_threshold = float(thresholds[best_idx])\n",
    "\n",
    "print(f'Optimal threshold (val set): {opt_threshold:.3f}')\n",
    "print(f'  F1:        {f1_scores[best_idx]:.3f}')\n",
    "print(f'  Precision: {precisions[best_idx]:.3f}')\n",
    "print(f'  Recall:    {recalls[best_idx]:.3f}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 11. Final Evaluation on Test Set"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Final evaluation — test set touched ONCE ──────────────────────────────────\n",
    "y_test_prob = best_model.predict_proba(X_test)[:,1]\n",
    "y_test_pred = (y_test_prob >= opt_threshold).astype(int)\n",
    "\n",
    "test_auc = roc_auc_score(y_test, y_test_prob)\n",
    "test_ap  = average_precision_score(y_test, y_test_prob)\n",
    "cm       = confusion_matrix(y_test, y_test_pred)\n",
    "tn,fp,fn,tp = cm.ravel()\n",
    "\n",
    "test_recall    = tp/(tp+fn) if (tp+fn)>0 else 0\n",
    "test_precision = tp/(tp+fp) if (tp+fp)>0 else 0\n",
    "test_f1        = 2*test_precision*test_recall/(test_precision+test_recall) \\\n",
    "                 if (test_precision+test_recall)>0 else 0\n",
    "\n",
    "print('='*60)\n",
    "print(f'FINAL TEST RESULTS — {best_name.upper()}')\n",
    "print('='*60)\n",
    "print(f'AUC-ROC:                  {test_auc:.4f}  {\"✅\" if test_auc>=MIN_AUC else \"❌\"}')\n",
    "print(f'Average Precision:        {test_ap:.4f}')\n",
    "print(f'Recall (discontinued):    {test_recall:.4f}  {\"✅\" if test_recall>=MIN_RECALL else \"❌\"}')\n",
    "print(f'Precision:                {test_precision:.4f}')\n",
    "print(f'F1 Score:                 {test_f1:.4f}')\n",
    "print(f'Decision threshold:       {opt_threshold:.3f}')\n",
    "print()\n",
    "print(f'True Negatives:   {tn:,}  |  False Positives: {fp:,}')\n",
    "print(f'False Negatives:  {fn:,}  |  True Positives:  {tp:,}')\n",
    "print()\n",
    "print(classification_report(y_test, y_test_pred, target_names=['Continued','Discontinued']))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Disaggregated evaluation — catch subgroup bias\n",
    "print('=== DISAGGREGATED EVALUATION ===')\n",
    "print('This checks whether the model is accurate across all subgroups.')\n",
    "print('A high overall AUC that hides a low AUC for young women is unacceptable.\\n')\n",
    "\n",
    "test_meta = features_encoded.loc[X_test.index, ['age_group','previous_method_category']].copy()\n",
    "test_meta['y_true'] = y_test.values\n",
    "test_meta['y_prob'] = y_test_prob\n",
    "\n",
    "print(f\"{'Subgroup':<35} {'AUC-ROC':>9} {'N':>7} {'Pos%':>6}\")\n",
    "print('-'*60)\n",
    "\n",
    "for group_col in ['age_group','previous_method_category']:\n",
    "    for group_val, gdf in test_meta.groupby(group_col, observed=True):\n",
    "        if len(gdf) < 50 or gdf['y_true'].nunique() < 2:\n",
    "            continue\n",
    "        try:\n",
    "            auc = roc_auc_score(gdf['y_true'], gdf['y_prob'])\n",
    "            flag = '' if auc >= MIN_AUC else '  ⚠️  BELOW THRESHOLD'\n",
    "            print(f\"  {str(group_val):<33} {auc:>9.3f} {len(gdf):>7,} \"\n",
    "                  f\"{gdf['y_true'].mean()*100:>5.0f}%{flag}\")\n",
    "        except Exception:\n",
    "            pass\n",
    "    print()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluation charts — ROC, PR, Confusion Matrix\n",
    "fig, axes = plt.subplots(1, 3, figsize=(16, 5))\n",
    "\n",
    "RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[0], name=best_name)\n",
    "axes[0].plot([0,1],[0,1],'k--',linewidth=0.8,label='Random (0.50)')\n",
    "axes[0].set_title('ROC Curve (Test Set)')\n",
    "axes[0].legend()\n",
    "\n",
    "p_t, r_t, _ = precision_recall_curve(y_test, y_test_prob)\n",
    "axes[1].plot(r_t, p_t, color=PALETTE['coral'], linewidth=2, label=f'AP={test_ap:.3f}')\n",
    "axes[1].axhline(y_test.mean(), color='k', linestyle='--', linewidth=0.8,\n",
    "                label=f'Baseline: {y_test.mean():.2f}')\n",
    "axes[1].axvline(test_recall, color=PALETTE['teal'], linestyle=':', linewidth=1.5,\n",
    "                label=f'Recall@threshold: {test_recall:.2f}')\n",
    "axes[1].set_xlabel('Recall')\n",
    "axes[1].set_ylabel('Precision')\n",
    "axes[1].set_title('Precision-Recall Curve (Test Set)')\n",
    "axes[1].legend(fontsize=9)\n",
    "\n",
    "ConfusionMatrixDisplay(cm, display_labels=['Continued','Discontinued']).plot(\n",
    "    ax=axes[2], cmap='Blues', colorbar=False\n",
    ")\n",
    "axes[2].set_title('Confusion Matrix (Test Set)')\n",
    "\n",
    "plt.suptitle(f'Final Evaluation — {best_name}', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "save_fig('12_final_evaluation')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 12. Feature Importance and SHAP"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "\n",
    "if hasattr(best_model, 'feature_importances_'):\n",
    "    imp_df = pd.DataFrame({\n",
    "        'feature': available_features,\n",
    "        'importance': best_model.feature_importances_\n",
    "    }).sort_values('importance', ascending=True)\n",
    "\n",
    "    imp_df.plot.barh(x='feature', y='importance', ax=ax,\n",
    "                     color=PALETTE['teal'], legend=False)\n",
    "    ax.set_title(f'Feature Importance — {best_name}')\n",
    "    ax.set_xlabel('Importance Score')\n",
    "    ax.set_ylabel('')\n",
    "\n",
    "    print('Top 5 features:')\n",
    "    for _, row in imp_df.tail(5)[::-1].iterrows():\n",
    "        print(f'  {row[\"feature\"]:<38} {row[\"importance\"]:.4f}')\n",
    "else:\n",
    "    coefs = pd.Series(np.abs(best_model.coef_[0]), index=available_features).sort_values()\n",
    "    coefs.plot.barh(ax=ax, color=PALETTE['teal'])\n",
    "    ax.set_title('Feature Coefficients (|β|) — Logistic Regression')\n",
    "\n",
    "plt.tight_layout()\n",
    "save_fig('13_feature_importance')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if SHAP_AVAILABLE and hasattr(best_model, 'feature_importances_'):\n",
    "    print('Computing SHAP values...')\n",
    "    sample = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_SEED)\n",
    "    try:\n",
    "        explainer   = shap.TreeExplainer(best_model)\n",
    "        shap_values = explainer.shap_values(sample)\n",
    "        sv = shap_values[1] if isinstance(shap_values, list) else shap_values\n",
    "\n",
    "        fig, axes = plt.subplots(1, 2, figsize=(14, 6))\n",
    "        plt.sca(axes[0])\n",
    "        shap.summary_plot(sv, sample, feature_names=available_features, show=False, plot_size=None)\n",
    "        axes[0].set_title('SHAP Beeswarm\\nRed=increases discontinuation risk')\n",
    "        plt.sca(axes[1])\n",
    "        shap.summary_plot(sv, sample, feature_names=available_features,\n",
    "                          plot_type='bar', show=False, plot_size=None)\n",
    "        axes[1].set_title('Mean |SHAP| Importance')\n",
    "        plt.tight_layout()\n",
    "        save_fig('14_shap_analysis')\n",
    "        print('SHAP complete.')\n",
    "    except Exception as e:\n",
    "        print(f'SHAP failed: {e}')\n",
    "else:\n",
    "    print('SHAP not available. Skipping.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 13. Model Persistence"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "timestamp   = datetime.now().strftime('%Y%m%d_%H%M')\n",
    "version_dir = MODELS_DIR / f'v{timestamp}'\n",
    "version_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "with open(version_dir/'chaguoai_discontinuation_model.pkl','wb') as f:\n",
    "    pickle.dump(best_model, f)\n",
    "\n",
    "with open(version_dir/'label_encoders.pkl','wb') as f:\n",
    "    pickle.dump(encoders, f)\n",
    "\n",
    "with open(version_dir/'feature_columns.json','w') as f:\n",
    "    json.dump(available_features, f, indent=2)\n",
    "\n",
    "metadata = {\n",
    "    'model_name':        best_name,\n",
    "    'model_version':     timestamp,\n",
    "    'trained_at':        datetime.now().isoformat(),\n",
    "    'training_data': {\n",
    "        'primary_dataset': 'Client_Service_Statistics.csv',\n",
    "        'geography':       'Siaya and Busia counties, Western Kenya',\n",
    "        'time_period':     '2013-2015',\n",
    "        'n_train':         int(len(X_train)),\n",
    "        'n_val':           int(len(X_val)),\n",
    "        'n_test':          int(len(X_test)),\n",
    "        'positive_rate':   float(y_train.mean()),\n",
    "        'population':      'Female FP clients aged 10-60',\n",
    "    },\n",
    "    'data_split': {'train':70,'validation':15,'test':15,'random_seed':RANDOM_SEED},\n",
    "    'test_set_performance': {\n",
    "        'auc_roc':                float(test_auc),\n",
    "        'average_precision':      float(test_ap),\n",
    "        'recall_discontinued':    float(test_recall),\n",
    "        'precision_discontinued': float(test_precision),\n",
    "        'f1_score':               float(test_f1),\n",
    "        'optimal_threshold':      float(opt_threshold),\n",
    "    },\n",
    "    'feature_columns':    available_features,\n",
    "    'categorical_features': CATEGORICAL_FEATURES,\n",
    "    'balance_strategy':   balance_strategy,\n",
    "    'target_variable': {\n",
    "        'name':      'discontinued',\n",
    "        'meaning_1': 'Client switched or stopped method at revisit',\n",
    "        'meaning_0': 'Client continued same method at revisit',\n",
    "    },\n",
    "    'known_limitations': [\n",
    "        'Trained on 2 counties in Western Kenya only (Siaya, Busia).',\n",
    "        'Does not capture women who stopped and never returned to clinic.',\n",
    "        'No attitudinal features (partner support, side-effect history) in CSS.',\n",
    "        '2013-2015 data may not reflect current method availability.',\n",
    "        'Disaggregated performance should be validated locally before deployment.',\n",
    "    ],\n",
    "    'recommended_use': (\n",
    "        'Rank recommended methods by predicted adherence probability. '\n",
    "        'Must be combined with WHO MEC safety filter. Never used standalone.'\n",
    "    ),\n",
    "}\n",
    "\n",
    "with open(version_dir/'model_metadata.json','w') as f:\n",
    "    json.dump(metadata, f, indent=2, default=str)\n",
    "\n",
    "print(f'Model saved: {version_dir}')\n",
    "for fname in ['chaguoai_discontinuation_model.pkl','label_encoders.pkl',\n",
    "              'feature_columns.json','model_metadata.json']:\n",
    "    print(f'  {fname}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 14. Inference Function"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_model(model_dir: Path) -> dict:\n",
    "    \"\"\"Load a saved model version. Returns dict with model, encoders, features, metadata.\"\"\"\n",
    "    model_dir = Path(model_dir)\n",
    "    return {\n",
    "        'model':     pickle.load(open(model_dir/'chaguoai_discontinuation_model.pkl','rb')),\n",
    "        'encoders':  pickle.load(open(model_dir/'label_encoders.pkl','rb')),\n",
    "        'features':  json.load(open(model_dir/'feature_columns.json')),\n",
    "        'metadata':  json.load(open(model_dir/'model_metadata.json')),\n",
    "    }\n",
    "\n",
    "\n",
    "def predict_discontinuation_risk(\n",
    "    user_profile: dict,\n",
    "    candidate_method: str,\n",
    "    model_bundle: dict,\n",
    ") -> dict:\n",
    "    \"\"\"\n",
    "    Predict discontinuation probability for one woman and one candidate method.\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    user_profile : dict\n",
    "        Keys: age_years, number_of_children, education_level,\n",
    "              fertility_intention, previous_method, counseled, county\n",
    "    candidate_method : str\n",
    "        Method name to evaluate e.g. 'Injectables', 'Implants', 'Condoms'\n",
    "    model_bundle : dict\n",
    "        Output from load_model().\n",
    "\n",
    "    Returns\n",
    "    -------\n",
    "    dict: method, discontinuation_probability, adherence_probability,\n",
    "          risk_level, threshold_used\n",
    "    \"\"\"\n",
    "    model      = model_bundle['model']\n",
    "    encs       = model_bundle['encoders']\n",
    "    feat_cols  = model_bundle['features']\n",
    "    threshold  = model_bundle['metadata']['test_set_performance']['optimal_threshold']\n",
    "\n",
    "    age      = user_profile.get('age_years', 25)\n",
    "    children = user_profile.get('number_of_children', 1)\n",
    "    prev_met = user_profile.get('previous_method', 'unknown')\n",
    "    curr_cat = categorize_method(candidate_method)\n",
    "    prev_cat = categorize_method(prev_met)\n",
    "\n",
    "    row = {\n",
    "        'age_years':                   age,\n",
    "        'number_of_children':          children,\n",
    "        'education_level_numeric':     EDUCATION_MAP.get(\n",
    "            user_profile.get('education_level','Primary Complete'), 2) or 2,\n",
    "        'fertility_intention_encoded': FERTILITY_MAP.get(\n",
    "            user_profile.get('fertility_intention','Later than 2 years'), 2) or 2,\n",
    "        'is_young_woman':              int(age < 20),\n",
    "        'is_older_woman':              int(age >= 40),\n",
    "        'has_high_parity':             int(children >= 5),\n",
    "        'is_nulliparous':              int(children == 0),\n",
    "        'adopted_long_acting':         int(curr_cat == 'long_acting_reversible'),\n",
    "        'was_on_long_acting':          int(prev_cat == 'long_acting_reversible'),\n",
    "        'wants_child_soon':            int(user_profile.get('fertility_intention','')=='Within 2 Years'),\n",
    "        'wants_no_more_children':      int(user_profile.get('fertility_intention','')=='No more Children'),\n",
    "        'was_counseled_binary':        int(user_profile.get('counseled', True)),\n",
    "        'previous_method_category':    prev_cat,\n",
    "        'current_method_category':     curr_cat,\n",
    "        'switch_direction':            switch_direction(curr_cat, prev_cat),\n",
    "        'county':                      user_profile.get('county','Unknown'),\n",
    "        'dataset_source':              user_profile.get('dataset_source','css_western_kenya_2013_2015'),\n",
    "    }\n",
    "\n",
    "    df_row = pd.DataFrame([row])\n",
    "\n",
    "    # Encode categoricals with saved encoders\n",
    "    for col in CATEGORICAL_FEATURES:\n",
    "        le  = encs.get(col)\n",
    "        if le is None: continue\n",
    "        val = str(df_row[col].iloc[0])\n",
    "        if val not in le.classes_: val = 'unknown'\n",
    "        df_row[col+'_encoded'] = le.transform([val])[0]\n",
    "\n",
    "    # Select features in the exact order the model was trained on\n",
    "    X_inf = df_row[[c for c in feat_cols if c in df_row.columns]].fillna(0)\n",
    "    disc_prob   = float(model.predict_proba(X_inf)[0,1])\n",
    "    adhere_prob = 1 - disc_prob\n",
    "\n",
    "    return {\n",
    "        'method':                      candidate_method,\n",
    "        'discontinuation_probability': round(disc_prob, 4),\n",
    "        'adherence_probability':       round(adhere_prob, 4),\n",
    "        'risk_level':                  'high' if disc_prob >= threshold else 'low',\n",
    "        'threshold_used':              round(threshold, 3),\n",
    "    }\n",
    "\n",
    "\n",
    "# Test inference\n",
    "bundle = load_model(version_dir)\n",
    "test_profile = {\n",
    "    'age_years':          22,\n",
    "    'number_of_children': 1,\n",
    "    'education_level':    'Primary Complete',\n",
    "    'fertility_intention':'Later than 2 years',\n",
    "    'previous_method':    'Injectables',\n",
    "    'counseled':          True,\n",
    "    'county':             'Siaya',\n",
    "}\n",
    "\n",
    "print('Inference test — 22-year-old woman, 1 child, previously on Injectables:')\n",
    "print(f'{\"Method\":<15} {\"Adherence\":>10} {\"Risk\":>6}')\n",
    "print('-'*35)\n",
    "for method in ['Injectables','Implants','Pills','Condoms','IUCD','BTL']:\n",
    "    r = predict_discontinuation_risk(test_profile, method, bundle)\n",
    "    print(f\"{method:<15} {r['adherence_probability']:>9.1%} {r['risk_level']:>6}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 15. Scalability and Retraining Architecture"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ──────────────────────────────────────────────────────────────────────────────\n",
    "# HOW TO ADD A NEW COUNTY OR COUNTRY\n",
    "#\n",
    "# The two main challenges when scaling to a new geography are:\n",
    "#\n",
    "# 1. COLUMN MISMATCH: A new CSV from Uganda may call the method column\n",
    "#    'method_chosen' instead of 'methodadopted'. The COLUMN_ALIAS_MAP\n",
    "#    below handles this without changing the core pipeline.\n",
    "#\n",
    "# 2. VALUE CODING MISMATCH: A new CSV may call the injectable\n",
    "#    'DMPA' instead of 'Injectables'. The METHOD_ALIAS_MAP handles this.\n",
    "#\n",
    "# Once both maps are filled in for the new CSV, call\n",
    "# retrain_with_new_data() — nothing else changes.\n",
    "# ──────────────────────────────────────────────────────────────────────────────\n",
    "\n",
    "# Add your country/study column mappings here\n",
    "COLUMN_ALIAS_MAP = {\n",
    "    'css_western_kenya': {\n",
    "        'age':              'age',\n",
    "        'noofchildren':     'noofchildren',\n",
    "        'educationlevel':   'educationlevel',\n",
    "        'fertilityintention':'fertilityintention',\n",
    "        'fpstatus':         'fpstatus',\n",
    "        'previousmethod':   'previousmethod',\n",
    "        'methodadopted':    'methodadopted',\n",
    "        'counseled':        'counseled',\n",
    "        'gender':           'gender',\n",
    "        'county':           'county',\n",
    "    },\n",
    "    # Example for a new country — fill in actual column names:\n",
    "    # 'anon_fp_system_nigeria': {\n",
    "    #     'age':              'client_age',\n",
    "    #     'noofchildren':     'parity',\n",
    "    #     'educationlevel':   'education',\n",
    "    #     'fertilityintention': 'wants_more_children',\n",
    "    #     'fpstatus':         'visit_type',\n",
    "    #     'previousmethod':   'last_method',\n",
    "    #     'methodadopted':    'current_method',\n",
    "    #     'counseled':        'was_counselled',\n",
    "    #     'gender':           'sex',\n",
    "    #     'county':           'state',\n",
    "    # },\n",
    "}\n",
    "\n",
    "# Add method name variations here\n",
    "METHOD_ALIAS_MAP = {\n",
    "    # Standard name → list of aliases found in other datasets\n",
    "    'Injectables': ['DMPA','Injectable','Injection','Injectables'],\n",
    "    'Implants':    ['Implant','Norplant','Jadelle','Implanon','Nexplanon'],\n",
    "    'Pills':       ['Pill','OCP','COC','POP','Daily Pill','Oral Pill'],\n",
    "    'Condoms':     ['Condom','Male Condom','Barrier'],\n",
    "    'IUCD':        ['IUD','Cu-IUD','Copper IUD','Coil'],\n",
    "    'BTL':         ['Sterilization','Female Sterilization','Tubal Ligation'],\n",
    "}\n",
    "\n",
    "\n",
    "def normalize_csv_for_pipeline(\n",
    "    df: pd.DataFrame,\n",
    "    dataset_key: str,\n",
    ") -> pd.DataFrame:\n",
    "    \"\"\"\n",
    "    Rename columns and normalize method values so any new CSV\n",
    "    can pass through build_features() without code changes.\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    df : pd.DataFrame  — raw new CSV\n",
    "    dataset_key : str  — key in COLUMN_ALIAS_MAP for this source\n",
    "    \"\"\"\n",
    "    alias = COLUMN_ALIAS_MAP.get(dataset_key)\n",
    "    if alias is None:\n",
    "        raise KeyError(\n",
    "            f'No column alias map found for \"{dataset_key}\".'\n",
    "            f' Add it to COLUMN_ALIAS_MAP.'\n",
    "        )\n",
    "\n",
    "    # Rename columns\n",
    "    reverse_alias = {v: k for k, v in alias.items()}\n",
    "    df = df.rename(columns=reverse_alias)\n",
    "\n",
    "    # Normalize method values\n",
    "    alias_lookup = {alias_val: std_name\n",
    "                    for std_name, aliases in METHOD_ALIAS_MAP.items()\n",
    "                    for alias_val in aliases}\n",
    "    for col in ['previousmethod', 'methodadopted']:\n",
    "        if col in df.columns:\n",
    "            df[col] = df[col].map(lambda v: alias_lookup.get(str(v).strip(), v))\n",
    "\n",
    "    return df\n",
    "\n",
    "\n",
    "def retrain_with_new_data(\n",
    "    new_data_path: Path,\n",
    "    existing_model_dir: Path,\n",
    "    new_dataset_key: str,\n",
    "    new_dataset_name: str,\n",
    "    combine_with_existing_X_train: pd.DataFrame = None,\n",
    "    combine_with_existing_y_train: pd.Series    = None,\n",
    ") -> Path:\n",
    "    \"\"\"\n",
    "    Retrain or fine-tune model with data from a new county or country.\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    new_data_path : Path\n",
    "        Path to new CSV.\n",
    "    existing_model_dir : Path\n",
    "        Directory of saved model version to fine-tune from.\n",
    "    new_dataset_key : str\n",
    "        Key in COLUMN_ALIAS_MAP for this source.\n",
    "    new_dataset_name : str\n",
    "        Identifier e.g. 'css_nairobi_2024'.\n",
    "    combine_with_existing_X_train : pd.DataFrame, optional\n",
    "        Pass the original X_train to combine old + new data.\n",
    "        If None, trains on new data only.\n",
    "\n",
    "    Returns\n",
    "    -------\n",
    "    Path to new saved model version.\n",
    "    \"\"\"\n",
    "    print(f'Retraining with: {new_data_path}')\n",
    "\n",
    "    new_raw = load_csv_safe(Path(new_data_path))\n",
    "    new_raw = normalize_csv_for_pipeline(new_raw, new_dataset_key)\n",
    "\n",
    "    if 'gender' in new_raw.columns:\n",
    "        new_raw = new_raw[new_raw['gender']=='Female']\n",
    "    new_raw = new_raw[\n",
    "        (new_raw['fpstatus']=='Revisit') &\n",
    "        new_raw['previousmethod'].notna() &\n",
    "        (new_raw['previousmethod']!='Missing')\n",
    "    ].copy()\n",
    "    new_raw['discontinued'] = (\n",
    "        new_raw['previousmethod'].str.strip().str.lower() !=\n",
    "        new_raw['methodadopted'].str.strip().str.lower()\n",
    "    ).astype(int)\n",
    "\n",
    "    new_feats = build_features(new_raw, dataset_name=new_dataset_name)\n",
    "    new_feats['discontinued'] = new_raw['discontinued'].values\n",
    "    new_feats_enc, new_encs = encode_categoricals(new_feats, CATEGORICAL_FEATURES, fit=True)\n",
    "\n",
    "    X_new = new_feats_enc[[c for c in available_features if c in new_feats_enc.columns]].fillna(0)\n",
    "    y_new = new_feats_enc['discontinued'].astype(int)\n",
    "\n",
    "    # Optionally combine with original training data to avoid forgetting\n",
    "    if combine_with_existing_X_train is not None:\n",
    "        # Take a 30% sample of original to avoid over-representing old data\n",
    "        n_sample = min(int(len(X_new) * 1.5), len(combine_with_existing_X_train))\n",
    "        X_old_sample = combine_with_existing_X_train.sample(n_sample, random_state=RANDOM_SEED)\n",
    "        y_old_sample = combine_with_existing_y_train.loc[X_old_sample.index]\n",
    "        X_new = pd.concat([X_new, X_old_sample], ignore_index=True)\n",
    "        y_new = pd.concat([y_new, y_old_sample], ignore_index=True)\n",
    "        print(f'Combined: {len(X_new):,} rows (new + existing sample)')\n",
    "\n",
    "    # Train same architecture as best model\n",
    "    import copy\n",
    "    new_model = copy.deepcopy(candidate_models.get(\n",
    "        best_name,\n",
    "        RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1)\n",
    "    ))\n",
    "    X_tr, X_te, y_tr, y_te = train_test_split(\n",
    "        X_new, y_new, test_size=0.2, random_state=RANDOM_SEED, stratify=y_new\n",
    "    )\n",
    "    new_model.fit(X_tr, y_tr)\n",
    "\n",
    "    if y_te.nunique() > 1:\n",
    "        new_auc = roc_auc_score(y_te, new_model.predict_proba(X_te)[:,1])\n",
    "        print(f'New model AUC on holdout: {new_auc:.3f}')\n",
    "\n",
    "    # Save new version\n",
    "    new_ts  = datetime.now().strftime('%Y%m%d_%H%M')\n",
    "    new_dir = MODELS_DIR / f'v{new_ts}_{new_dataset_name}'\n",
    "    new_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "    with open(new_dir/'chaguoai_discontinuation_model.pkl','wb') as f:\n",
    "        pickle.dump(new_model, f)\n",
    "    with open(new_dir/'label_encoders.pkl','wb') as f:\n",
    "        pickle.dump(new_encs, f)\n",
    "    with open(new_dir/'feature_columns.json','w') as f:\n",
    "        json.dump(available_features, f)\n",
    "\n",
    "    print(f'New model saved: {new_dir}')\n",
    "    return new_dir\n",
    "\n",
    "\n",
    "def log_chatbot_interaction(\n",
    "    user_profile: dict,\n",
    "    method_used: str,\n",
    "    outcome: str,          # 'continued' or 'discontinued'\n",
    "    chatbot_log_path: Path,\n",
    "    retrain_threshold: int = 500,\n",
    ") -> int:\n",
    "    \"\"\"\n",
    "    Log one chatbot follow-up interaction as a labeled training example.\n",
    "    Triggers a retraining notice when retrain_threshold examples accumulate.\n",
    "\n",
    "    This is how live chatbot data feeds back into model improvement.\n",
    "    Call this at Day-90 follow-up when user confirms they continued or stopped.\n",
    "    \"\"\"\n",
    "    record = {\n",
    "        'age':               user_profile.get('age_years'),\n",
    "        'noofchildren':      user_profile.get('number_of_children'),\n",
    "        'educationlevel':    user_profile.get('education_level'),\n",
    "        'fertilityintention':user_profile.get('fertility_intention'),\n",
    "        'previousmethod':    user_profile.get('previous_method', method_used),\n",
    "        'methodadopted':     method_used if outcome=='continued' else 'Switched',\n",
    "        'fpstatus':          'Revisit',\n",
    "        'counseled':         'Yes',\n",
    "        'county':            user_profile.get('county', 'Unknown'),\n",
    "        'gender':            'Female',\n",
    "        'discontinued':      1 if outcome=='discontinued' else 0,\n",
    "        'data_source':       'chatbot_followup',\n",
    "        'logged_at':         datetime.now().isoformat(),\n",
    "    }\n",
    "\n",
    "    log_path = Path(chatbot_log_path)\n",
    "    df_new   = pd.DataFrame([record])\n",
    "    if log_path.exists():\n",
    "        existing = pd.read_csv(log_path)\n",
    "        combined = pd.concat([existing, df_new], ignore_index=True)\n",
    "    else:\n",
    "        combined = df_new\n",
    "    combined.to_csv(log_path, index=False)\n",
    "\n",
    "    n = len(combined)\n",
    "    if n >= retrain_threshold and n % retrain_threshold == 0:\n",
    "        print(f'\\n⚠️  RETRAIN TRIGGER: {n} chatbot examples logged.')\n",
    "        print(f'   Run retrain_with_new_data() using: {log_path}')\n",
    "    return n\n",
    "\n",
    "\n",
    "print('Scalability functions ready.')\n",
    "print()\n",
    "print('To scale to new county/country:')\n",
    "print('  1. Add entry to COLUMN_ALIAS_MAP with actual column names')\n",
    "print('  2. Add method name variations to METHOD_ALIAS_MAP if needed')\n",
    "print('  3. Call retrain_with_new_data(new_csv, version_dir, new_key, new_name)')\n",
    "print()\n",
    "print('To use chatbot data for retraining:')\n",
    "print('  Call log_chatbot_interaction() at every Day-90 follow-up')\n",
    "print('  System triggers retraining notice at 500-example intervals')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 16. Design Decisions and Full Documentation\n",
    "\n",
    "---\n",
    "\n",
    "### Why Client_Service_Statistics.csv and not Final_women_Data_ANON.csv\n",
    "\n",
    "The women survey has 1,997 women total. Of those, only 514 ever used FP, and the direct discontinuation signal — the q334 reason-for-stopping columns — applies to a subset of those. Training a machine learning model on fewer than 500 labeled examples produces high-variance estimates: small changes in the training set produce large swings in predicted probabilities. The model would memorise the training data rather than learning generalisable patterns.\n",
    "\n",
    "Client Service Statistics has 79,885 female revisit records with valid prior method information. Crucially, the signal is observational: we do not ask a woman to recall whether she stopped — we see it directly. She arrived on Injectables and left on Implants. That is a recorded discontinuation, not a recalled one. Recalled behaviour is systematically distorted by social desirability bias and memory error. Observed behaviour is not.\n",
    "\n",
    "The women survey is not wasted. It provides clinical context — the q341 non-use reasons confirm that side effects and partner opposition are the dominant barriers, which validates our counseling and partner-support features. We use it for EDA and to triangulate the patterns we see in the CSS data.\n",
    "\n",
    "---\n",
    "\n",
    "### Why XGBoost / LightGBM over deep learning\n",
    "\n",
    "On structured tabular data with under 100,000 rows and under 20 features, gradient boosting consistently outperforms neural networks. This is reproducible across hundreds of Kaggle medical datasets and is supported by the academic literature (Gorishniy et al., 2021; Grinsztajn et al., 2022). Neural networks require substantially more data to overcome the inductive biases of tree-based methods on this type of problem. The additional complexity of a neural network introduces no accuracy benefit here and significantly reduces interpretability — a critical property for a clinical health tool.\n",
    "\n",
    "We also compared Logistic Regression as the interpretable baseline. If LR performs within 2 percentage points of XGBoost on AUC-ROC, we prefer LR for its transparency. A clinical reviewer can inspect coefficients. They cannot inspect boosted trees.\n",
    "\n",
    "---\n",
    "\n",
    "### Feature Selection Rationale\n",
    "\n",
    "Every feature in this model has a clinical or behavioural justification and must be available at ChaguoAI intake time:\n",
    "\n",
    "**age_years** — The WHO MEC uses age to define safety thresholds for combined hormonal methods. Age also correlates with life stage, which predicts method preference and stability.\n",
    "\n",
    "**number_of_children** — Parity shapes method choice (nulliparous women face different IUD insertion considerations) and is one of the strongest predictors of whether a woman wants long-acting versus short-acting methods.\n",
    "\n",
    "**education_level_numeric** — Higher education correlates with better method-use self-efficacy: understanding how to use methods correctly, knowing how to seek care when problems arise, and being more likely to continue through temporary side effects.\n",
    "\n",
    "**fertility_intention_encoded** — The single strongest predictor of method continuation. A woman who wants a child within two years will not sustain a five-year implant regardless of her other characteristics.\n",
    "\n",
    "**previous_method_category** — Past behaviour predicts future behaviour. A woman who previously used short-acting hormonal methods has a documented higher discontinuation risk than one who previously used long-acting methods.\n",
    "\n",
    "**current_method_category** — Long-acting reversible contraceptives (implants, IUDs) have inherently lower discontinuation rates because there is no daily or monthly action required. This is the most clinically significant method-level predictor.\n",
    "\n",
    "**switch_direction** — Captures the directionality of method change, which has distinct clinical meaning. A woman downgrading from an implant to condoms is a different risk profile than one upgrading from pills to an implant.\n",
    "\n",
    "**was_counseled_binary** — The Kenya FP Guidelines 7th Edition and the WHO SPR both show that counselling reduces discontinuation by preparing women for expected side effects. A woman who is told that irregular bleeding is normal in the first three months is far more likely to continue than one who is not.\n",
    "\n",
    "**is_young_woman / is_older_woman** — These derived flags capture non-linear age effects. Young women under 20 have higher discontinuation due to life-stage instability (school, relationship changes). Women over 40 have different reproductive goals and method preferences.\n",
    "\n",
    "**county / dataset_source** — Geographic context captures population-level heterogeneity in method availability, CHW coverage, facility access, and cultural norms. Critically, dataset_source allows the model to learn country-specific patterns when trained on multi-country data, without those patterns contaminating the individual-level clinical features.\n",
    "\n",
    "Features deliberately excluded: facility name (not available at chatbot intake), organisation type (not clinically meaningful for individual prediction), year and month (temporal leakage risk — we do not want the model to learn that 2015 clients behave differently from 2013 clients as a predictive feature).\n",
    "\n",
    "---\n",
    "\n",
    "### Target Variable Definition\n",
    "\n",
    "A female revisit client who reported a previous method and adopted a different method is labeled discontinued=1. We excluded removal visits (Removal_Implant, Removal_IUCD) from the switched group because these are planned terminations at the end of a method's lifespan, not unplanned discontinuations. Labeling a woman who completed five years of implant use as a discontinuer would misrepresent her experience and corrupt the training signal.\n",
    "\n",
    "Known limitation: we do not capture women who stopped using contraception entirely and never returned to a facility. Our dataset contains only women who came back. This is a selection bias toward women with some level of continued engagement with the health system. Women who experienced the worst side effects and dropped out entirely are invisible in this data. The model will underestimate discontinuation rates for the most severe adverse-event scenarios.\n",
    "\n",
    "---\n",
    "\n",
    "### Why 70/15/15 and Not 80/20\n",
    "\n",
    "With 79,885 records, a 15% test set gives approximately 11,983 test examples — sufficient for evaluation confidence intervals within ±0.5 percentage points on AUC. The additional 5% relative to an 80/10/10 split goes to the validation set, which enables more stable model selection and finer threshold optimisation. The test set is touched exactly once at final evaluation. Any decision based on test set performance — including the choice of this split ratio — would constitute data leakage.\n",
    "\n",
    "---\n",
    "\n",
    "### Bias Mitigation Strategy\n",
    "\n",
    "Three measures are in place:\n",
    "\n",
    "**Disaggregated evaluation** — After every training run, AUC-ROC is computed separately for each age group and each method category. A model with overall AUC 0.78 that has AUC 0.58 for women under 20 is unacceptable. The disaggregated check catches this before deployment.\n",
    "\n",
    "**Geographic context as a feature** — The dataset_source feature encodes where the data came from. This prevents the model from confusing geographic patterns with individual clinical signals. When trained on multi-country data, the model learns to separate \"this woman is likely to discontinue\" from \"this country has lower overall continuation rates.\"\n",
    "\n",
    "**Model card with explicit limitations** — The metadata.json documents that this model was trained on two counties in Western Kenya in 2013-2015. Any deployment in a different geography requires local validation before clinical use. This is not optional documentation — it is a clinical safety requirement.\n",
    "\n",
    "---\n",
    "\n",
    "### Scalability Plan\n",
    "\n",
    "Phase 1 (New county, same country): Add the county's column mapping to COLUMN_ALIAS_MAP. Call retrain_with_new_data() with combine_with_existing_X_train to merge old and new data. The model retains knowledge of the original population while adapting to local patterns. The dataset_source feature automatically differentiates the two populations.\n",
    "\n",
    "Phase 2 (New country): Add country-specific column and value mappings. Run full retrain on combined data. Evaluate disaggregated performance by country to confirm the model generalises correctly.\n",
    "\n",
    "Phase 3 (Chatbot feedback loop): Every Day-90 follow-up interaction where a woman confirms she continued or stopped is logged via log_chatbot_interaction(). When 500 new labeled examples accumulate, retrain_with_new_data() is called automatically. This creates a continuous learning loop where the model improves as ChaguoAI is used. Because chatbot users are a self-selected population (they sought FP information proactively), their data is tagged with data_source='chatbot_followup' and modeled as a separate context — preventing their behavioural patterns from being treated as representative of the general population."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
NOTEBOOK_EOF
echo "Notebook written successfully"
Output

Notebook written successfully
Done


